# AWS FraudShield — Exploratory Data Analysis (EDA)
## Real-Time E-Commerce Behavioral Fraud Detection

This notebook explores the transaction dataset, examining key characteristics of e-commerce fraud:
1. **Severe Class Imbalance**: Fraud represents a small fraction of overall volume (~5%).
2. **Behavioral Deviations**: How transaction amount relative to user historical average serves as a strong signal.
3. **Device & Location Anomalies**: High correlation between account takeover attempts and new hardware/unusual IPs.
4. **Legitimate Outliers**: Distinguishing legitimate high-value purchases (e.g. buying a laptop on known device) from fraudulent spikes.

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 20)
print("Libraries successfully imported.")

### 1. Load Data and User Baselines

In [ ]:
DATA_PATH = "../data/fraud_transactions.csv"
PROFILES_PATH = "../models/user_profiles.json"

# Fallback to current directory if running inside notebooks/
if not os.path.exists(DATA_PATH):
    DATA_PATH = "data/fraud_transactions.csv"
    PROFILES_PATH = "models/user_profiles.json"

df = pd.read_csv(DATA_PATH)
with open(PROFILES_PATH, 'r') as f:
    user_profiles = json.load(f)

print(f"Loaded {len(df)} transactions.")
df.head()

### 2. Class Distribution & Imbalance Analysis

In financial fraud detection, class imbalance is extreme. Traditional accuracy ($(\text{TP} + \text{TN}) / \text{Total}$) is misleading because predicting all transactions as legitimate would achieve ~95% accuracy while missing 100% of fraud.

In [ ]:
fraud_counts = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].value_counts(normalize=True) * 100

print("Class Distribution:")
for label, count in fraud_counts.items():
    name = "Fraudulent (1)" if label == 1 else "Legitimate (0)"
    print(f"  {name}: {count:,} ({fraud_pct[label]:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#3b82f6', '#ef4444']
ax.bar(['Legitimate (0)', 'Fraud (1)'], fraud_counts.values, color=colors, edgecolor='none', width=0.5)
ax.set_ylabel('Transaction Count')
ax.set_title('Transaction Volume by Class')
for i, v in enumerate(fraud_counts.values):
    ax.text(i, v + 100, f"{v:,} ({fraud_pct.iloc[i]:.1f}%)", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 3. Transaction Amount Comparison: Legitimate vs Fraudulent

In [ ]:
print("Transaction Amount Statistics (INR ₹):")
print(df.groupby('is_fraud')['amount'].describe().T)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot (Log Scale)
sns.boxplot(data=df, x='is_fraud', y='amount', palette=['#3b82f6', '#ef4444'], ax=ax1)
ax1.set_yscale('log')
ax1.set_xticklabels(['Legitimate (0)', 'Fraud (1)'])
ax1.set_title('Transaction Amount (Log Scale)')
ax1.set_ylabel('Amount (₹)')

# Attack Scenario breakdown
scenario_counts = df[df['is_fraud'] == 1]['fraud_scenario'].value_counts()
ax2.barh(scenario_counts.index, scenario_counts.values, color='#ef4444')
ax2.set_title('Fraud Attack Vectors in Dataset')
ax2.set_xlabel('Count')

plt.tight_layout()
plt.show()

### 4. Behavioral Feature Engineering Analysis

Let's observe how comparing transaction amount against the user's specific historical baseline reveals anomalies that absolute amount cannot show.

In [ ]:
# Compute amount-to-average ratio
user_avg_map = {uid: prof['avg_amount'] for uid, prof in user_profiles.items()}
df['user_avg_amount'] = df['user_id'].map(user_avg_map).fillna(850.0)
df['amount_ratio'] = df['amount'] / df['user_avg_amount']

plt.figure(figsize=(10, 4))
sns.kdeplot(data=df[df['is_fraud'] == 0]['amount_ratio'], label='Legitimate', color='#3b82f6', fill=True, clip=(0, 20))
sns.kdeplot(data=df[df['is_fraud'] == 1]['amount_ratio'], label='Fraudulent', color='#ef4444', fill=True, clip=(0, 20))
plt.axvline(x=4.0, color='gray', linestyle='--', label='4x Heuristic Threshold')
plt.title('Distribution of Transaction Amount / User Baseline Ratio')
plt.xlabel('Amount Ratio (x times user historical average)')
plt.legend()
plt.tight_layout()
plt.show()

### 5. Conclusions from EDA
- Fraud cannot be caught by single-column thresholds alone. Some legitimate users buy ₹30,000 laptops, while card-testing fraudsters spend ₹50 on micro-transactions.
- Multidimensional signals (Device unfamiliarity + Location mismatch + Amount spike + Temporal anomaly) provide the strongest separation.
- Tree-based models (Random Forest, XGBoost) and regularized Logistic Regression are well-suited to capture these non-linear feature interactions.